## Experiment 1: EDA and Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("Toyato1a.csv")
print("Shape:", df.shape)
num = df.select_dtypes(include="number").columns
cat = df.select_dtypes(include="object").columns
print("Numerical:", len(num), "| Categorical:", len(cat))
print(df[num].agg(["min", "max", "mean"]))
print(df["Age"].describe())

sns.boxplot(x=df["Age"]); plt.show()

q1, q3 = df["Guarantee_Period"].quantile([.25, .75])
iqr = q3 - q1
print("IQR outliers:", df.index[(df["Guarantee_Period"] < q1-1.5*iqr) | (df["Guarantee_Period"] > q3+1.5*iqr)].tolist())

gp = df["Guarantee_Period"]
z = (gp - gp.mean()) / gp.std()
print("Z-score outliers:", df.index[z.abs() > 3].tolist())

features = ["Age", "Price", "Quarterly_Tax", "Weight"]
corr = df[features].corr()
print(corr)
pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
print("Highest positive:", pairs.idxmax(), pairs.max())
print("Highest negative:", pairs.idxmin(), pairs.min())
print("Correlations > 0.70:", pairs[pairs.abs() > .70])

print("cc skewness:", df["cc"].skew())
sns.histplot(df["cc"], kde=True); plt.show()
sns.countplot(data=df, x="Fuel_Type"); plt.show()
sns.violinplot(y=df["Age"]); plt.show()
sns.scatterplot(data=df, x="Age", y="Price"); plt.show()
sns.boxplot(data=df, x="Fuel_Type", y="Age"); plt.show()
print(pd.crosstab(df["Fuel_Type"], df["Gears"]))
sns.countplot(data=df, x="Fuel_Type", hue="Gears"); plt.show()
sns.pairplot(df[["Age", "Price", "KM", "Weight", "Fuel_Type"]], hue="Fuel_Type"); plt.show()

df = pd.read_csv("Toyato1b.csv")
missing = df.isna().mean() * 100
print("Missing (%):", missing[missing > 0])
df["Age"] = df["Age"].fillna(df["Age"].mean())
df["KM"] = df["KM"].interpolate(method="nearest")
df["Fuel_Type"] = df["Fuel_Type"].fillna(df["Fuel_Type"].mode()[0])
df.drop(columns=missing[missing > 10].index, inplace=True, errors="ignore")

if "Guarantee_Period" in df:
    gp = df["Guarantee_Period"]
    z = (gp - gp.mean()) / gp.std()
    df = df[z.abs().le(3) | z.isna()]

print("Duplicate percentage:", df.duplicated().mean() * 100)
df = df.drop_duplicates().drop_duplicates(subset=["Price", "Age", "Mfg_Month", "Fuel_Type"])
tax = df["Quarterly_Tax"]
df["Tax_MinMax"] = (tax - tax.min()) / (tax.max() - tax.min())
df["Tax_ZScore"] = (tax - tax.mean()) / tax.std()
df["Fuel_Type_LE"] = LabelEncoder().fit_transform(df["Fuel_Type"].astype(str))
df = pd.get_dummies(df, columns=["Fuel_Type"])


## Experiment 2: Feature Selection and Extraction

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_regression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from PIL import Image

df = pd.read_csv("B3_toyato_feature.csv")
data = df.drop(columns=["BOVAG_Guarantee", "Doors"], errors="ignore").copy()

num = data.select_dtypes(include="number")
outliers = []
for col in num:
    q1, q3 = num[col].quantile([.25, .75])
    if ((num[col] < q1-1.5*(q3-q1)) | (num[col] > q3+1.5*(q3-q1))).any():
        outliers.append(col)
data.drop(columns=outliers, inplace=True, errors="ignore")

low_var = data.select_dtypes(include="number").var()
data.drop(columns=low_var[low_var < 10].index, inplace=True, errors="ignore")
corr = data.select_dtypes(include="number").corr()["Price"].abs()
data.drop(columns=corr[corr < .2].index, inplace=True, errors="ignore")

corr = data.drop(columns="Price").select_dtypes(include="number").corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
data.drop(columns=[c for c in upper if (upper[c] > .8).any()], inplace=True, errors="ignore")

X = data.drop(columns="Price").select_dtypes(include="number")
y = data["Price"]
X_pos = X - X.min() + 1
selected = SelectKBest(chi2, k=min(10, X.shape[1])).fit_transform(X_pos, y)
mi = mutual_info_regression(X, y)
print("Highest mutual information:", X.columns[np.argmax(mi)])

X_std = (X - X.mean()) / X.std()
cov = np.cov(X_std.T)
vals, vecs = np.linalg.eigh(cov)
X_pca = X_std.to_numpy() @ vecs[:, np.argsort(vals)[::-1][:2]]

y_bins = pd.qcut(y, q=3, labels=False, duplicates="drop")
X_lda = LinearDiscriminantAnalysis(n_components=min(2, len(np.unique(y_bins))-1)).fit_transform(X, y_bins)
print("Image pixel mean:", np.array(Image.open("Lenna.png").convert("L")).mean())


## Experiment 3: Regression

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("iris_regression.csv")
df = df.fillna(df.mean(numeric_only=True))

X, y = df["SepalLengthCm"].to_numpy(), df["SepalWidthCm"].to_numpy()
b1 = np.sum((X-X.mean())*(y-y.mean())) / np.sum((X-X.mean())**2)
b0 = y.mean() - b1*X.mean()
pred = b0 + b1*X
sse = np.sum((y-pred)**2)
print("SSE:", sse, "RMSE:", np.sqrt(sse/len(y)), "R2:", 1-sse/np.sum((y-y.mean())**2))
order = np.argsort(X)
plt.scatter(X, y); plt.plot(X[order], pred[order], color="red"); plt.show()
print("Prediction at 5.5:", b0+b1*5.5)

Xm = df[["SepalLengthCm", "PetalLengthCm"]].to_numpy()
ym = df["SepalWidthCm"].to_numpy()
design = np.c_[np.ones(len(Xm)), Xm]
beta = np.linalg.lstsq(design, ym, rcond=None)[0]
pm = design @ beta
sse_m = np.sum((ym-pm)**2)
print("Multiple regression RMSE:", np.sqrt(sse_m/len(ym)), "R2:", 1-sse_m/np.sum((ym-ym.mean())**2))

Xlog = df[["SepalWidthCm", "SepalLengthCm", "PetalWidthCm", "PetalLengthCm"]].to_numpy()
ylog = (df["Species"] == df["Species"].iloc[0]).astype(int).to_numpy()
Xlog = (Xlog-Xlog.mean(axis=0))/(Xlog.std(axis=0)+1e-12)
w, b, lr, costs = np.zeros(Xlog.shape[1]), 0., .01, []
for _ in range(500):
    p = 1/(1+np.exp(-np.clip(Xlog@w+b, -500, 500)))
    costs.append(-np.mean(ylog*np.log(p+1e-15)+(1-ylog)*np.log(1-p+1e-15)))
    w -= lr*(Xlog.T@(p-ylog))/len(ylog)
    b -= lr*np.mean(p-ylog)
plt.plot(costs); plt.title("Logistic Regression Cost"); plt.show()


## Experiment 4: Single Layer Perceptron

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv("penguins_SLP.csv")
df = df.fillna(df.mean(numeric_only=True))
X = StandardScaler().fit_transform(df[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]])
labels = LabelEncoder().fit_transform(df["species"])
mask = labels < 2
X, y = X[mask], np.where(labels[mask] == 0, -1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)

w, b, lr, errors = np.zeros(X_train.shape[1]), 0., .01, []
for _ in range(100):
    error = 0
    for xi, target in zip(X_train, y_train):
        pred = 1 if xi@w+b >= 0 else -1
        if pred != target:
            w += lr*target*xi
            b += lr*target
            error += 1
    errors.append(error)

plt.plot(errors); plt.xlabel("Epoch"); plt.ylabel("Errors"); plt.show()
pred = np.where(X_test@w+b >= 0, 1, -1)
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print("Accuracy:", accuracy_score(y_test, pred))


## Experiment 5: PyTorch MLP

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

data = load_digits()
X = StandardScaler().fit_transform(data.data)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)

Xt, yt = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)
Xv = torch.tensor(X_test, dtype=torch.float32)

model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(),
                      nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(),
                      nn.Linear(16, 10))
loss_fn, optimizer = nn.CrossEntropyLoss(), optim.Adam(model.parameters(), lr=.01)
losses = []
for _ in range(100):
    optimizer.zero_grad()
    loss = loss_fn(model(Xt), yt)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.plot(losses); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.show()
with torch.no_grad():
    pred = model(Xv).argmax(dim=1).numpy()
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print("Accuracy:", accuracy_score(y_test, pred))


## Experiment 6: Hierarchical Agglomerative Clustering

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

df = pd.read_csv("customer_clustering.csv").select_dtypes(include="number").dropna()
X = StandardScaler().fit_transform(df)
dendrogram(linkage(X, method="single", metric="euclidean")); plt.show()

labels = {}
for k in (2, 3):
    model = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels[k] = model.fit_predict(X)
    plt.scatter(X[:, 0], X[:, 1], c=labels[k]); plt.title(f"Clusters k={k}"); plt.show()

scores = {k: silhouette_score(X, lab) for k, lab in labels.items()}
best = max(scores, key=scores.get)
print("Silhouette scores:", scores)
df["Cluster_Label"] = labels[best]
df.to_csv("HAC_Tableau_Export.csv", index=False)


## Experiment 7: Fuzzy C-Means

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from fcmeans import FCM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

df = pd.read_csv("driverFCM.csv").select_dtypes(include="number")
df = df.fillna(df.mean())
X = StandardScaler().fit_transform(df)
plt.scatter(X[:, 0], X[:, 1]); plt.show()

models, scores = [], []
for k in range(2, 6):
    model = FCM(n_clusters=k, m=2, max_iter=150, random_state=42)
    model.fit(X)
    labels = model.predict(X)
    models.append((model, labels))
    scores.append(silhouette_score(X, labels))

plt.plot(range(2, 6), scores); plt.xlabel("Clusters"); plt.ylabel("Silhouette"); plt.show()
best, labels = models[int(np.argmax(scores))]
print("Centroids:\n", best.centers)
df["FCM_Cluster"] = labels
df.to_csv("FCM_Tableau_Export.csv", index=False)


## Experiment 8: Self-Organizing Map

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from minisom import MiniSom
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

df = pd.read_csv("wineFCM.csv").select_dtypes(include="number")
df = df.fillna(df.mean())
X = StandardScaler().fit_transform(df)
plt.scatter(X[:, 0], X[:, 1]); plt.show()

models, scores = [], []
for k in range(2, 6):
    som = MiniSom(1, k, X.shape[1], sigma=1, learning_rate=.5, random_seed=42)
    som.random_weights_init(X)
    som.train_random(X, 100)
    weights = som.get_weights().reshape(-1, X.shape[1])
    labels = np.array([np.linalg.norm(x-weights, axis=1).argmin() for x in X])
    models.append(labels)
    scores.append(silhouette_score(X, labels))

plt.plot(range(2, 6), scores); plt.xlabel("Grid size (1 x k)"); plt.ylabel("Silhouette"); plt.show()
labels = models[int(np.argmax(scores))]
df["SOM_Cluster"] = labels
df.to_csv("SOM_PowerBI_Export.csv", index=False)


## Experiment 9: Apriori Association Rules

In [ ]:
import pandas as pd
from itertools import combinations

df = pd.read_csv("Shop1.csv")
transactions = [set(row.dropna()) for _, row in df.iterrows()]
min_support, min_confidence = .08, .5
n = len(transactions)

counts = {}
for transaction in transactions:
    for item in transaction:
        key = frozenset([item])
        counts[key] = counts.get(key, 0) + 1
frequent = {itemset: count/n for itemset, count in counts.items() if count/n >= min_support}

# Generate pairs from frequent single items
items = [next(iter(itemset)) for itemset in frequent]
for pair in combinations(items, 2):
    itemset = frozenset(pair)
    support = sum(itemset.issubset(t) for t in transactions)/n
    if support >= min_support:
        frequent[itemset] = support

rules = []
for itemset, support in frequent.items():
    if len(itemset) > 1:
        for item in itemset:
            antecedent = frozenset([item])
            confidence = support / frequent[antecedent]
            if confidence >= min_confidence:
                rules.append((antecedent, itemset-antecedent, confidence))
print("Frequent itemsets:", frequent)
print("Rules:", rules)


## Experiment 10: Random Forest and AdaBoost

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv("iris_SLP.csv")
df = df.fillna(df.mean(numeric_only=True))
X = df[["SepalWidthCm", "SepalLengthCm", "PetalWidthCm", "PetalLengthCm"]]
y = LabelEncoder().fit_transform(df["Species"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)

for n in (10, 50, 100):
    for depth in (3, 5, None):
        model = RandomForestClassifier(n_estimators=n, max_depth=depth, random_state=42)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        print(f"Random Forest n={n}, depth={depth}: accuracy={accuracy_score(y_test, pred):.2f}")
        print(confusion_matrix(y_test, pred))

for n in (10, 50, 100):
    for lr in (.01, .1, 1.):
        model = AdaBoostClassifier(n_estimators=n, learning_rate=lr, random_state=42)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        print(f"AdaBoost n={n}, lr={lr}: accuracy={accuracy_score(y_test, pred):.2f}")
        print(confusion_matrix(y_test, pred))
